In [9]:
# ================================
# (1) Import Libraries
# ================================
import requests
import json
import re
import os
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit
from collections import defaultdict

# ================================
# (2) Config
# ================================
API_KEY            = API_KEY_LLM
API_URL            = API_URL_LLM
MODEL              = LLM_Model
DEBUG_API_RESPONSE = False
USE_CACHE          = False
CACHE_FILE         = "Files/Log_txt_files/"
BATCH_SIZE         = 5        # Number of tables per API call — tune down to 3 if still slow
API_TIMEOUT        = 180      # Seconds per API call

# ================================
# (3) Load Bronze Metadata Table
# ================================
bronze_df = spark.table("Bronze.table_column_details")

# FIX: Use limit + cache to avoid full scan repeatedly
bronze_df.cache()
pdf = bronze_df.toPandas()
data = pdf.to_dict(orient="records")

# Group columns by TableName
table_schema = defaultdict(list)
for row in data:
    table_schema[row["TableName"]].append({
        "ColumnName":     row["ColumnName"],
        "ColumnDataType": row["ColumnDataType"]
    })

# Convert to list format
structured_data = [
    {"TableName": table, "columns": cols}
    for table, cols in table_schema.items()
]

# FIX: Diagnostic print — tells you upfront how large the job is
total_tables  = len(structured_data)
total_columns = sum(len(t["columns"]) for t in structured_data)
sample_prompt_size = len(json.dumps(structured_data[:BATCH_SIZE], indent=2))

print("=== Bronze Schema Loaded ===")
print(f"  Total Tables   : {total_tables}")
print(f"  Total Columns  : {total_columns}")
print(f"  Batch Size     : {BATCH_SIZE} tables/call")
print(f"  Total Batches  : {-(-total_tables // BATCH_SIZE)} (ceil)")
print(f"  Sample Prompt  : ~{sample_prompt_size} chars per batch")
print()

# ================================
# (4) Batch Helper
# ================================
def chunk_tables(data_list, batch_size):
    """Split structured_data into smaller batches for parallel API calls."""
    for i in range(0, len(data_list), batch_size):
        yield data_list[i : i + batch_size]

# ================================
# (5) Prompt Builder
# ================================
def build_prompt(formatted_json):
    prompt = f"""
You are a STRICT Data Vault 2.0 Architect specializing in Supply Chain & Inventory Management.

Your task is to transform MULTIPLE independent table schemas into VALID Data Vault 2.0 structures.

IMPORTANT:
- Process EACH table independently
- DO NOT create cross-table links unless explicitly implied within the SAME table
- Be deterministic and consistent

---------------------------------------
DOMAIN CONTEXT (SUPPLY CHAIN)
---------------------------------------
Entities include:
- MASTER DATA: Suppliers, Products, Employees, Categories, Warehouses, Stores, Customers
- TRANSACTIONAL: Purchase Orders, GRN, Invoices, Payments
- INVENTORY: Stock, Transactions, Adjustments, Transfers
- SALES: POS Sales

---------------------------------------
INPUT TABLE SCHEMA
---------------------------------------
{formatted_json}

---------------------------------------
MANDATORY RULES
---------------------------------------

✔ RULE 1: HUB (Business Keys ONLY)
- HUB represents core business entity
- MUST contain ONLY business keys (no descriptive attributes)
- Prefer NATURAL keys:
  - *_code → PRIMARY business key (preferred)
  - *_id   → technical key (use ONLY if no natural key exists)
- One HUB per entity
- Naming: HUB_<SINGULAR_ENTITY_NAME>

✔ RULE 2: LINK (Strict Rule)
- Create LINK ONLY if:
  - Table contains 2 or more FOREIGN KEYS referencing DIFFERENT entities
- LINK connects HUBs
- Naming: LINK_<ENTITY1>_<ENTITY2>

DO NOT create LINK for:
- Single foreign key
- Self-referencing keys (e.g., parent_id)

✔ RULE 3: SATELLITE (All Descriptive Data)
- Contains ALL non-key attributes:
  - names, descriptions, emails
  - timestamps (created_at, updated_at)
  - status flags (is_active)
  - numeric values (price, quantity, amount)
- Also contains:
  - foreign keys when NOT part of a LINK
  - self-referencing keys (parent_id)
- Naming: SAT_<PARENT_ENTITY>

✔ RULE 4: SELF-REFERENCING KEYS
- Example: parent_id
- Treat as SATELLITE attribute
- DO NOT create LINK

✔ RULE 5: COLUMN CLASSIFICATION
- *_code → HUB (business key)
- *_id:
  → if primary identifier → HUB
  → if referencing another entity:
      - part of multi-FK → LINK
      - otherwise → SATELLITE
- *_name, *_desc, *_status, *_date, *_at → SATELLITE
- numeric fields → SATELLITE

✔ RULE 6: DATA VAULT METADATA (MANDATORY)
- HUB must include:
  - hash_key (PK)
  - load_date
  - record_source

- LINK must include:
  - hash_key (PK)
  - load_date
  - record_source

- SATELLITE must include:
  - parent_hash_key (FK to HUB/LINK)
  - load_date
  - record_source

✔ RULE 7: CONSISTENCY
- Use SAME naming pattern across all tables
- Singular entity names only

---------------------------------------
OUTPUT FORMAT (STRICT JSON ONLY)
---------------------------------------

Return ONLY a valid JSON array. No explanation. No markdown. No extra text.

[
  {{
    "Bronze_TableName":  "TableName",
    "Bronze_ColumnName": "columnname",
    "Bronze_DataType":   "datatype",

    "Silver_TableName":  "HUB_ENTITY or LINK_ENTITY1_ENTITY2 or SAT_ENTITY",
    "Silver_ColumnName": "column_name",
    "Silver_DataType":   "datatype",

    "IsPrimaryKey": true/false,
    "IsForeignKey": true/false,
    "DV_ObjectType": "HUB or LINK or SATELLITE"
  }}
]

---------------------------------------
VALIDATION CHECKS (MANDATORY)
---------------------------------------

✔ Every column must appear exactly ONCE
✔ No duplicates
✔ HUB contains ONLY business keys
✔ LINK contains ONLY foreign keys
✔ SATELLITE contains ALL descriptive attributes
✔ Self-referencing keys go to SATELLITE
✔ JSON must be valid

---------------------------------------
BEGIN TRANSFORMATION
---------------------------------------

Analyze EACH table independently and return ONLY the JSON array.
"""
    return prompt

# ================================
# (6) OpenRouter API Call
# ================================
def call_openrouter(prompt, batch_index=None):
    """Call the OpenRouter API with retry logic and progress feedback."""
    label = f"Batch {batch_index}" if batch_index is not None else "Request"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model":       MODEL,
        "messages":    [{"role": "user", "content": prompt}],
        "temperature": 0,
    }

    max_retries = 2
    for attempt in range(1, max_retries + 1):
        try:
            print(f"  [{label}] API call attempt {attempt}/{max_retries}...")
            response = requests.post(
                API_URL,
                headers=headers,
                json=payload,
                timeout=API_TIMEOUT
            )
            response.raise_for_status()
            response_json = response.json()

            if DEBUG_API_RESPONSE:
                print(f"DEBUG API response [{label}]:")
                print(json.dumps(response_json, indent=2))

            content = response_json["choices"][0]["message"]["content"]
            print(f"  [{label}] ✅ Response received ({len(content)} chars)")
            print(content)
            return content

        except requests.exceptions.Timeout:
            print(f"  [{label}] ⚠️  Timeout on attempt {attempt}. Retrying...")
            if attempt == max_retries:
                raise TimeoutError(f"{label}: API timed out after {max_retries} attempts.")

        except (KeyError, IndexError, TypeError) as exc:
            raise ValueError(f"Unexpected API response structure from OpenRouter [{label}].") from exc

# ================================
# (7) JSON Extractor
# ================================
def extract_json_array(text, batch_index=None):
    """Parse and extract a JSON array from raw LLM output."""
    label = f"Batch {batch_index}" if batch_index is not None else "Response"
    if not text:
        raise ValueError(f"[{label}] Model returned empty content.")

    cleaned = text.strip()
    cleaned = re.sub(r"^```json\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"^```\s*",     "", cleaned)
    cleaned = re.sub(r"\s*```$",     "", cleaned)

    try:
        parsed = json.loads(cleaned)
        if isinstance(parsed, list):
            return parsed
    except json.JSONDecodeError:
        pass

    match = re.search(r"\[\s*\{.*\}\s*\]", cleaned, re.DOTALL)
    if not match:
        raise ValueError(f"[{label}] Could not find a valid JSON array in model output.")

    return json.loads(match.group(0))

# ================================
# (8) Cache Helpers
# ================================
def load_cached_response(cache_key="default"):
    cache_path = f"{CACHE_FILE}_{cache_key}.txt"
    if not USE_CACHE or not os.path.exists(cache_path):
        return None
    with open(cache_path, "r", encoding="utf-8") as f:
        return f.read()

def save_cached_response(content, cache_key="default"):
    if not USE_CACHE:
        return
    cache_path = f"{CACHE_FILE}_{cache_key}.txt"
    with open(cache_path, "w", encoding="utf-8") as f:
        f.write(content)

# ================================
# (9) Save to Spark Delta Table
# ================================
def save_to_raw_vault_table(rows, table_name="Bronze.Metadata_Raw_Vault"):
    """
    Persist LLM-parsed JSON rows into a managed Delta table in the Raw Vault layer.

    Args:
        rows       : List of dicts from extract_json_array()
        table_name : Fully qualified Delta table name (schema.table)
    """
    if not rows:
        raise ValueError("No rows to save. LLM returned empty or unparseable output.")

    # Step 1: Pandas DataFrame for easy schema inference
    result_pdf = pd.DataFrame(rows)

    # Step 2: Normalize column names — strip whitespace, replace spaces
    result_pdf.columns = [col.strip().replace(" ", "_") for col in result_pdf.columns]

    # Step 3: Cast boolean columns correctly (LLM may return strings)
    for bool_col in ["IsPrimaryKey", "IsForeignKey"]:
        if bool_col in result_pdf.columns:
            result_pdf[bool_col] = result_pdf[bool_col].map(
                lambda x: True if str(x).strip().lower() == "true" else False
            )

    # Step 4: Pandas → Spark DataFrame
    spark_df = spark.createDataFrame(result_pdf)

    # Step 5: Add Data Vault audit metadata columns
    batch_id = pd.Timestamp.now().strftime("%Y%m%d%H%M%S")
    spark_df = (
        spark_df
        .withColumn("dv_load_date",     current_timestamp())
        .withColumn("dv_record_source", lit("LLM_OpenRouter"))
        .withColumn("dv_batch_id",      lit(batch_id))
    )

    # Step 6: Write as managed Delta table
    (
        spark_df.write
        .format("delta")
        .mode("overwrite")                  # Use "append" to retain history across runs
        .option("overwriteSchema", "true")  # Allows schema evolution between runs
        .saveAsTable(table_name)
    )

    # FIX: Use len(rows) instead of spark_df.count() — avoids expensive re-scan
    row_count = len(rows)
    print(f"\n✅ Successfully saved {row_count} rows → `{table_name}`")
    print(f"   Batch ID     : {batch_id}")
    print(f"   Record Source: LLM_OpenRouter")
    print(f"   Write Mode   : overwrite\n")
    spark_df.show(10, truncate=False)

    return spark_df

# ================================
# (10) Main Orchestrator
# ================================
def main():
    all_rows    = []
    failed_batches = []

    batches     = list(chunk_tables(structured_data, BATCH_SIZE))
    total       = len(batches)

    print(f"🚀 Starting LLM pipeline: {total} batches × ~{BATCH_SIZE} tables each\n")

    for i, batch in enumerate(batches, start=1):
        table_names = [t["TableName"] for t in batch]
        print(f"--- Batch {i}/{total} | Tables: {table_names} ---")

        batch_json = json.dumps(batch, indent=2)

        try:
            # Check cache first (per-batch key)
            cache_key       = f"batch_{i}"
            cached_response = load_cached_response(cache_key)

            if cached_response:
                print(f"  ⚡ Using cached response for batch {i}")
                raw_output = cached_response
            else:
                raw_output = call_openrouter(build_prompt(batch_json), batch_index=i)
                save_cached_response(raw_output, cache_key)

            if DEBUG_API_RESPONSE:
                print(f"\n===== RAW RESPONSE (Batch {i}) =====")
                print(raw_output)
                print(f"===== END RAW (Batch {i}) =====\n")

            rows = extract_json_array(raw_output, batch_index=i)
            print(f"  Parsed {len(rows)} rows from batch {i}")
            all_rows.extend(rows)

        except Exception as e:
            print(f"  ❌ Batch {i} FAILED: {e}")
            failed_batches.append({"batch_index": i, "tables": table_names, "error": str(e)})
            continue  # Skip failed batch, process remaining

    # --- Summary ---
    print(f"\n{'='*50}")
    print(f"✅ Completed  : {total - len(failed_batches)}/{total} batches")
    print(f"❌ Failed     : {len(failed_batches)} batches")
    print(f"📦 Total Rows : {len(all_rows)}")
    if failed_batches:
        print("\nFailed batches detail:")
        for fb in failed_batches:
            print(f"  Batch {fb['batch_index']} — {fb['tables']} — Error: {fb['error']}")
    print(f"{'='*50}\n")

    if not all_rows:
        raise RuntimeError("All batches failed. No rows to save. Check API key, model, or network.")

    # --- Save all rows to Delta table ---
    save_to_raw_vault_table(all_rows, table_name="Bronze.Metadata_Raw_Vault")

    # --- Optional: Show parsed JSON (can be noisy for large runs) ---
    if DEBUG_API_RESPONSE:
        print("===== ALL PARSED JSON =====")
        print(json.dumps(all_rows, indent=2))
        print("===== END PARSED JSON =====\n")

# ================================
# (11) Entry Point
# ================================
main()

StatementMeta(, 3ece1c8e-d0e9-4abb-afe4-a7420dc4295a, 11, Finished, Available, Finished, False)

=== Bronze Schema Loaded ===
  Total Tables   : 27
  Total Columns  : 217
  Batch Size     : 5 tables/call
  Total Batches  : 6 (ceil)
  Sample Prompt  : ~3899 chars per batch

🚀 Starting LLM pipeline: 6 batches × ~5 tables each

--- Batch 1/6 | Tables: ['categories', 'customers', 'employees', 'goods_receipt_notes', 'grn_items'] ---
  [Batch 1] API call attempt 1/2...
  [Batch 1] ✅ Response received (13054 chars)
[
  {
    "Bronze_TableName": "categories",
    "Bronze_ColumnName": "category_code",
    "Bronze_DataType": "string",
    "Silver_TableName": "HUB_CATEGORY",
    "Silver_ColumnName": "category_code",
    "Silver_DataType": "string",
    "IsPrimaryKey": true,
    "IsForeignKey": false,
    "DV_ObjectType": "HUB"
  },
  {
    "Bronze_TableName": "categories",
    "Bronze_ColumnName": "category_id",
    "Bronze_DataType": "int",
    "Silver_TableName": "HUB_CATEGORY",
    "Silver_ColumnName": "category_id",
    "Silver_DataType": "int",
    "IsPrimaryKey": false,
    "IsForeignK

In [12]:
%%sql
select * from Bronze.Metadata_Raw_Vault;

StatementMeta(, 3ece1c8e-d0e9-4abb-afe4-a7420dc4295a, 14, Finished, Available, Finished, False)

<Spark SQL result set with 217 rows and 12 fields>